# Supermarket Sales Analysis
## IBM SkillsBuild Data Analytics with AI Internship 2026

**Project:** Supermarket Sales Analysis  
**Dataset:** 500 supermarket sales transactions

### Objective
Analyze supermarket sales data to understand product performance, branch performance, category sales, payment preferences, customer spending, and customer ratings, then convert the findings into business insights and decisions.

### Tools
- Python
- Pandas
- Matplotlib
- Seaborn
- Jupyter Notebook / Google Colab
- Excel/CSV dataset

> **Dataset setup:** Place the provided supermarket dataset in the same folder as this notebook. The loading cell automatically looks for an Excel or CSV file if the default filename is not found.


In [ ]:
# Install dependencies if needed (run once in a new environment)
# !pip install pandas matplotlib seaborn openpyxl


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Try a common filename first; otherwise detect the first Excel/CSV dataset in the folder.
DATA_FILE = "supermarket_sales.xlsx"

if Path(DATA_FILE).exists():
    file_path = Path(DATA_FILE)
else:
    candidates = [
        p for p in Path(".").glob("*")
        if p.suffix.lower() in [".xlsx", ".xls", ".csv"]
        and not p.name.startswith("~$")
    ]
    if not candidates:
        raise FileNotFoundError(
            "No Excel/CSV dataset found. Place the supermarket dataset in the same folder "
            "as this notebook or update DATA_FILE."
        )
    file_path = candidates[0]

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)
else:
    df = pd.read_excel(file_path)

print(f"Loaded: {file_path}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()


In [ ]:
# Basic structure and data types
df.info()


In [ ]:
# Check missing values and duplicate rows
missing_values = df.isna().sum()
duplicate_rows = df.duplicated().sum()

print("Missing values by column:")
display(missing_values.to_frame("Missing Values"))

print(f"\nDuplicate rows: {duplicate_rows}")


In [ ]:
# Convert Date and verify numeric columns
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_columns = ["Quantity", "Unit Price", "Rating", "Sales"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Date conversion complete.")
print(df[numeric_columns].describe())


In [ ]:
# Calculate sales independently and compare with the supplied Sales column
df["Calculated Sales"] = df["Quantity"] * df["Unit Price"]
df["Sales Difference"] = df["Sales"] - df["Calculated Sales"]

max_difference = df["Sales Difference"].abs().max()

print(f"Maximum absolute difference: {max_difference:.10f}")
if max_difference < 1e-6:
    print("Validation passed: Sales = Quantity × Unit Price for all transactions.")
else:
    print("Warning: Some Sales values differ from Quantity × Unit Price.")


## Key Business Questions

In [ ]:
# 1. Which product generates the highest sales?
product_sales = df.groupby("Product")["Sales"].sum().sort_values(ascending=False)
top_product = product_sales.index[0]
top_product_sales = product_sales.iloc[0]

print(f"Highest-selling product: {top_product}")
print(f"Sales: ₹{top_product_sales:,.2f}")


In [ ]:
# 2. Which branch performs best?
branch_sales = df.groupby(["Branch", "City"])["Sales"].sum().sort_values(ascending=False)
best_branch = branch_sales.index[0]
best_branch_sales = branch_sales.iloc[0]

print(f"Best-performing branch: Branch {best_branch[0]} ({best_branch[1]})")
print(f"Sales: ₹{best_branch_sales:,.2f}")


In [ ]:
# 3. Which category sells the most?
category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
top_category = category_sales.index[0]
top_category_sales = category_sales.iloc[0]

print(f"Highest-selling category: {top_category}")
print(f"Sales: ₹{top_category_sales:,.2f}")


In [ ]:
# 4. What is the most popular payment method?
payment_counts = df["Payment"].value_counts()
popular_payment = payment_counts.index[0]
popular_payment_count = payment_counts.iloc[0]

print(f"Most popular payment method: {popular_payment}")
print(f"Transactions: {popular_payment_count}")


In [ ]:
# 5. Do Members spend more than Normal customers?
customer_type_avg = df.groupby("Customer Type")["Sales"].mean().sort_values(ascending=False)

for customer_type, avg_sales in customer_type_avg.items():
    print(f"{customer_type}: ₹{avg_sales:,.2f}")

if customer_type_avg.get("Member", 0) > customer_type_avg.get("Normal", 0):
    print("\nMembers spend more on average.")
else:
    print("\nNormal customers spend more on average.")


In [ ]:
# 6. What is the average customer rating?
average_rating = df["Rating"].mean()
print(f"Average customer rating: {average_rating:.2f} / 5")


## Visualizations

In [ ]:
# Product-wise sales
plt.figure(figsize=(10, 6))
product_sales.sort_values().plot(kind="barh")
plt.title("Sales by Product")
plt.xlabel("Sales (₹)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


In [ ]:
# Branch-wise sales
branch_city_sales = df.groupby(["Branch", "City"])["Sales"].sum().sort_values(ascending=False)
labels = [f"Branch {b} ({c})" for b, c in branch_city_sales.index]

plt.figure(figsize=(9, 5))
plt.bar(labels, branch_city_sales.values)
plt.title("Sales by Branch")
plt.xlabel("Branch")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Category-wise sales
plt.figure(figsize=(10, 6))
category_sales.sort_values().plot(kind="barh")
plt.title("Sales by Category")
plt.xlabel("Sales (₹)")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


In [ ]:
# Payment method usage
plt.figure(figsize=(8, 5))
payment_counts.sort_values().plot(kind="barh")
plt.title("Payment Method Usage")
plt.xlabel("Number of Transactions")
plt.ylabel("Payment Method")
plt.tight_layout()
plt.show()


In [ ]:
# Customer type average spending
plt.figure(figsize=(7, 5))
customer_type_avg.plot(kind="bar")
plt.title("Average Transaction Value by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Average Sales (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Monthly sales trend
monthly_sales = df.groupby(df["Date"].dt.to_period("M"))["Sales"].sum()

plt.figure(figsize=(10, 5))
monthly_sales.plot(marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Business Insights and Decisions

In [ ]:
insights = {
    "Top Product": f"{top_product} with ₹{top_product_sales:,.2f} in sales",
    "Best Branch": f"Branch {best_branch[0]} ({best_branch[1]}) with ₹{best_branch_sales:,.2f} in sales",
    "Top Category": f"{top_category} with ₹{top_category_sales:,.2f} in sales",
    "Popular Payment": f"{popular_payment} with {popular_payment_count} transactions",
    "Average Rating": f"{average_rating:.2f}/5",
}

for k, v in insights.items():
    print(f"{k}: {v}")

print("\nBusiness decisions:")
print("1. Keep sufficient stock of high-selling products and categories.")
print("2. Study the operational or customer factors behind Branch C's strong performance.")
print("3. Continue supporting UPI as a major payment option.")
print("4. Improve customer service and experience to increase ratings.")
print("5. Use customer spending patterns when designing membership offers.")


## Conclusion

The analysis converts raw supermarket transaction data into practical business insights. It identifies the strongest products, branches, categories, payment preferences, customer spending patterns, and overall customer satisfaction. These findings can support inventory planning, branch-level strategy, payment support, customer-service improvements, and membership offers.
